# Notebook 4. AWQ

AWQ (Lin et al., MLSys 2024) starts from a measurement rather than an objective: a small fraction of a layer's input channels carry activations far larger than the rest, and the rounding errors on the weights of those channels account for most of the output error. Instead of compensating errors after the fact, as GPTQ does, AWQ scales those weight columns up before rounding and folds the inverse scale into the layer that produces the input, so the network computes the same function with a smaller error where it matters. This notebook measures the effect of a single scale, implements the scale search, checks it against `pare.schemes.awq`, and runs AWQ on the smallest model Pare's layerwise path supports.

Prerequisites are Notebooks 1 to 3.

In [ ]:
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Salient input channels

The salience of input channel $j$ is measured by its mean absolute activation over the calibration data, $\bar{x}_j = \frac{1}{n}\sum_t \lvert x_{tj} \rvert$. The cell below builds a calibration set in which five of 256 channels have activations twenty times larger than the others, which is the kind of pattern the AWQ paper reports in OPT and Llama models, and recovers them from $\bar{x}$.

In [ ]:
n_in, n_out = 256, 64
salient = [10, 47, 89, 134, 200]


def make_inputs(n, gain=20.0):
    X = torch.randn(n, n_in) * 0.3
    X[:, salient] *= gain
    return X


torch.manual_seed(0)
X_calib = make_inputs(4096)
x_mean = X_calib.abs().mean(dim=0)                  # [n_in]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(range(n_in), x_mean.numpy(), width=1)
ax.set_xlabel("input channel")
ax.set_ylabel("mean |x|")
ax.set_title("Per-channel mean absolute activation")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()
print("five largest channels:", sorted(x_mean.topk(5).indices.tolist()))

## 2. What one scale does to one group

For a linear layer $Y = X W^T$, any positive per-channel scale $s \in \mathbb{R}^{I}$ leaves the output unchanged if it is applied to the weights and removed from the inputs,

$$Y = X W^T = \left(X\,\mathrm{diag}(s)^{-1}\right)\left(W\,\mathrm{diag}(s)\right)^T.$$

AWQ quantizes $W' = W\,\mathrm{diag}(s)$ and lets the layer that produces $X$ absorb $\mathrm{diag}(s)^{-1}$ (Section 5). The gain comes from how group quantization sets its step. Within a group of 128 weights, the step is $(\max - \min) / 15$, so it is set by the largest-magnitude weights in the group. Scaling one salient column by $s > 1$ changes the step only if that column becomes the group's extreme. While it does not, the rounding error of the scaled weight stays at most half a step, and dividing by $s$ afterwards shrinks it by $s$. Once the column does become the extreme, the step grows for all 128 weights.

The cell quantizes 4000 random groups with the weight in column 10 scaled by $s$, and reports the mean step relative to RTN and the mean error on column 10 and on the other 127 columns.

In [ ]:
def grid(bits):
    return (-128, 127) if bits == 8 else (0, 2**bits - 1)


def rtn_groups(W, bits=4, group_size=128):
    # Asymmetric per-group RTN (Notebook 2). Returns the dequantized weights and the steps.
    q_min, q_max = grid(bits)
    n_rows, n_cols = W.shape
    Wg = W.reshape(n_rows, -1, group_size)
    w_min = Wg.amin(dim=2, keepdim=True).clamp(max=0.0)
    w_max = Wg.amax(dim=2, keepdim=True).clamp(min=0.0)
    step = ((w_max - w_min) / (q_max - q_min)).clamp(min=1e-8)
    zero = torch.round(q_min - w_min / step).clamp(q_min, q_max)
    q = torch.clamp(torch.round(Wg / step + zero), q_min, q_max)
    return (step * (q - zero)).reshape(n_rows, n_cols), step


torch.manual_seed(0)
groups = torch.randn(4000, 128) * 0.05
base, step_rtn = rtn_groups(groups)
others = torch.arange(128) != 10
print("   s   step/step_rtn   MAE column 10   MAE other columns")
for s_val in (1.0, 1.5, 2.0, 3.0, 4.0, 8.0):
    s = torch.ones(128)
    s[10] = s_val
    W_hat, step = rtn_groups(groups * s)
    W_hat = W_hat / s                                 # back in the original units
    err = (W_hat - groups).abs()
    print(f"{s_val:4.1f}   {(step / step_rtn).mean():8.2f}      {err[:, 10].mean():.5f}"
          f"         {err[:, others].mean():.5f}")

Up to $s = 2$ the step barely moves and the error on the scaled column halves. Beyond that the scaled weight is often the group's extreme, the step grows, and the other 127 columns pay for it. The scale search in the next section finds the balance for a whole layer.

## 3. Choosing the scales

AWQ ties the scale of each channel to its salience through one exponent, $s_j = \bar{x}_j^{\alpha}$, and searches $\alpha$ on a grid from 0 (no scaling, plain RTN) towards 1 (scale in proportion to activation). The implementation in `llm-awq`, which Pare follows, divides $s$ by $\sqrt{s_{\max} s_{\min}}$ so that the scales are centred on 1, and uses the grid $\alpha \in \{0, 0.05, \ldots, 0.95\}$.

The paper picks $\alpha$ by the output error on the calibration inputs, $\lVert X W^T - X \hat{W}^T\rVert^2$ with $\hat{W} = Q(W\,\mathrm{diag}(s))\,\mathrm{diag}(s)^{-1}$. Pare uses a cheaper proxy that needs no forward pass, the weight error weighted by salience, $\mathrm{mean}\big((\hat{W} - W)^2 \,\bar{x}\big)$. The cell implements both and compares them with RTN on a layer fed by the inputs of Section 1.

In [ ]:
def awq_scales(alpha, x_mean):
    s = x_mean.clamp(min=1e-6).pow(alpha).clamp(min=1e-4)
    return s / (s.max() * s.min()).sqrt()


def awq_search(W, x_mean, loss, n_grid=20):
    # Returns the dequantized weights, the chosen alpha and the scales.
    best = None
    for i in range(n_grid):
        alpha = i / n_grid
        s = awq_scales(alpha, x_mean)
        W_hat = rtn_groups(W * s)[0] / s
        value = loss(W_hat, W)
        if best is None or value < best[0]:
            best = (value, alpha, s, W_hat)
    return best[3], best[1], best[2]


torch.manual_seed(1)
W = torch.randn(n_out, n_in) * 0.05
X_test = make_inputs(1000)
Y = X_test @ W.T


def output_error(W_hat):
    return ((Y - X_test @ W_hat.T).norm() / Y.norm()).item()


def output_loss(W_hat, W):            # the paper's objective, on the calibration inputs
    return (X_calib @ W_hat.T - X_calib @ W.T).pow(2).mean().item()


def proxy_loss(W_hat, W):             # Pare's objective
    return ((W_hat - W).pow(2) * x_mean).mean().item()


W_rtn = rtn_groups(W)[0]
W_awq_out, alpha_out, _ = awq_search(W, x_mean, output_loss)
W_awq_proxy, alpha_proxy, s_proxy = awq_search(W, x_mean, proxy_loss)
for label, W_hat in [("RTN INT4 g=128", W_rtn),
                     (f"AWQ, output loss, alpha {alpha_out:.2f}", W_awq_out),
                     (f"AWQ, proxy loss, alpha {alpha_proxy:.2f}", W_awq_proxy)]:
    print(f"{label:>32}: relative output error {output_error(W_hat):.4f}")

The output-loss search picks the larger $\alpha$ and recovers more of RTN's error. The proxy cannot see how the errors of different columns combine in the output, so it stops at a smaller $\alpha$ and recovers part of the gain; the printed lines show how much that costs on this layer.

The next cell checks that `pare.schemes.awq._search_scale` returns the proxy-loss scales computed above.

In [ ]:
from pare import QuantConfig
from pare.schemes.awq import _search_scale

linear = torch.nn.Linear(n_in, n_out, bias=False)
linear.weight.data = W.clone()
config = QuantConfig(bits=4, scheme="awq", granularity="per_group", group_size=128)
s_pare = _search_scale([linear], x_mean, config)
assert torch.allclose(s_pare, s_proxy), "scales differ"
print("awq_search with the proxy loss matches pare.schemes.awq._search_scale")

## 4. Where the error moves

Per column, AWQ lowers the error on the salient channels and raises it slightly elsewhere. Weighted by the activations each column multiplies, the trade is favourable.

In [ ]:
err_rtn = (W_rtn - W).abs().mean(dim=0)
err_awq = (W_awq_out - W).abs().mean(dim=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(n_in), err_rtn.numpy(), width=1, alpha=0.7, label="RTN")
axes[0].bar(range(n_in), err_awq.numpy(), width=1, alpha=0.7, label="AWQ")
axes[0].set_title("mean weight error per input channel")
axes[0].set_xlabel("input channel")
axes[0].legend()
axes[1].bar(range(n_in), (err_rtn * x_mean).numpy(), width=1, alpha=0.7, label="RTN")
axes[1].bar(range(n_in), (err_awq * x_mean).numpy(), width=1, alpha=0.7, label="AWQ")
axes[1].set_title("weight error times mean |x|")
axes[1].set_xlabel("input channel")
axes[1].legend()
plt.tight_layout()
plt.show()
others_mask = torch.ones(n_in, dtype=torch.bool)
others_mask[salient] = False
for label, mask in [("salient channels", salient), ("other channels", others_mask)]:
    print(f"{label:>16}: RTN error {err_rtn[mask].mean():.5f}, "
          f"AWQ error {err_awq[mask].mean():.5f}")

## 5. Where the inverse scale goes

$\mathrm{diag}(s)^{-1}$ has to be applied to $X$ before the linear layer, and AWQ puts it into the operator that produces $X$. In a Llama-style block there are four such pairs, and `pare.schemes.awq._apply_awq_groups` handles each.

| Scaled linears | Operator that absorbs $1/s$ | Pare helper |
|---|---|---|
| `q_proj`, `k_proj`, `v_proj` | `input_layernorm` weight (and bias) | `_scale_ln_fcs` |
| `o_proj` | `v_proj` output rows | `_scale_fc_fc` |
| `gate_proj`, `up_proj` | the norm that feeds the MLP | `_scale_ln_fcs` |
| `down_proj` | `up_proj` output rows | `_scale_fc_fc` |

A norm absorbs $1/s$ because its output is an elementwise affine map of the normalized input, and a preceding linear absorbs it because its output rows can be rescaled. The fusion is only valid when that operator feeds the scaled linears directly. Pare skips the `v_proj` to `o_proj` pair for grouped-query attention, where the two have different widths, and whenever a norm sits between them; it uses `pre_feedforward_layernorm` on sandwich-norm blocks; and on post-norm blocks without `input_layernorm` it skips AWQ for the block and quantizes it with RTN, with a warning.

## 6. AWQ and GPTQ on the 7B models

The README reports both methods on five models with the same calibration data and evaluation. AWQ has the lower perplexity on Llama-2-7B, Llama-3-8B and Llama-3.1-8B, GPTQ has the lower perplexity on Qwen2.5-7B and OLMo-3-7B, and the ranking by zero-shot accuracy differs from the ranking by perplexity. Neither method is ahead on every model, and the two differ in a way that matters for deployment: AWQ needs a block structure it can fuse scales into, while GPTQ works on any linear layer.

## 7. AWQ in Pare

Pare's AWQ runs block by block over `model.model.layers`. GPT-2 has no such structure, so Pare would fall back to RTN on it; the smallest model with a Llama-style block is Qwen2.5-0.5B (about 1 GB to download). The cell quantizes it with RTN, GPTQ and AWQ at INT4 with groups of 128, all from the same 8 calibration sequences of 512 tokens, and evaluates WikiText-2 perplexity on 20 windows of 512 tokens. On a laptop CPU this takes a few minutes.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from pare import quantize
from pare.calibration.data import load_wikitext2_calibration
from pare.eval.perplexity import evaluate_perplexity

model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
calib = load_wikitext2_calibration(tokenizer, n_samples=8, seq_len=512)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT4 g=128": QuantConfig(bits=4, scheme="rtn", granularity="per_group", group_size=128),
    "GPTQ INT4 g=128": QuantConfig(bits=4, scheme="gptq", granularity="per_group", group_size=128),
    "AWQ INT4 g=128": QuantConfig(bits=4, scheme="awq", granularity="per_group", group_size=128),
}
for label, cfg in configs.items():
    model = fresh_model()
    quantize(model, cfg, calibration_data=calib, device="cpu")
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>16}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

The printed deltas show what each calibrated method recovers of RTN's INT4 loss on this model with eight calibration sequences. The ordering of GPTQ and AWQ here is one data point on one small model; Section 6 describes how it varies across the 7B models, whose numbers in the README use 128 sequences of 2048 tokens.

## Exercises

1. In Section 2, scale two columns at once, 10 and 11, by the same $s$. At what $s$ does the step start to grow, and why is it lower than with one column?
2. In Section 3, set `gain=3` in `make_inputs` and rerun. How much does AWQ gain over RTN when the salient channels are only mildly salient, and which $\alpha$ is chosen?
3. Print the `input_layernorm.weight` of the first block of Qwen2.5-0.5B before and after `quantize` in Section 7. What do the changes tell you about the scales AWQ chose for `q_proj`, `k_proj` and `v_proj`?

## Summary

| Idea | Formula or setting | Role |
|------|--------------------|------|
| Salience | $\bar{x}_j = \frac{1}{n}\sum_t \lvert x_{tj} \rvert$ | Which columns matter |
| Scaling identity | $X W^T = (X s^{-1})(W s)^T$ | Output unchanged, rounding changed |
| Scale | $s_j = \bar{x}_j^{\alpha} / \sqrt{s_{\max} s_{\min}}$ | One exponent per layer group |
| Search | 20 values of $\alpha$ in $[0, 1)$ | Output loss in the paper, weighted weight loss in Pare |
| Fusion | norm or preceding linear absorbs $1/s$ | No extra operator at inference |

Notebook 5 applies the same scaling identity in the other direction: SmoothQuant moves difficulty out of the activations so that they can be quantized to INT8 as well.